# Modeling the Feedback Prior

**MODELING notebook — lift formulas, empirical-Bayes centering, pool-relative scaling, evidence discipline, and the aggression tradeoff**

This is a **modeling** notebook: every section introduces one modeling choice and evaluates it.

Every section follows *Question → What we do → Figure/Table → Reading → Artifact → Caveat*.
Each code cell states what it does and each output is interpreted in the following
cell, so a reader with no access to the code can follow the reasoning. All numbers are
read from immutable artifacts in `results/` through `paper_lib`; missing optional
experiments print `PENDING` with their producer command instead of failing.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
import paper_lib as L

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 130)
pd.set_option("display.width", 200)
np.random.seed(42)

PRIMARY_RUN = "M4_intersection_dev_conditioned_continuous"
BLIND_RUN = "M4_intersection_dev_blind_continuous"
GATE_FEATURE_COLS = ["top1_faiss", "top5_mean_faiss", "top5_min_faiss", "top5_std_faiss",
                     "retrieval_margin", "top5_spread", "retrieval_overlap", "max_lift",
                     "mean_lift", "n_positive_lifts", "n_negative_lifts", "lift_conflict",
                     "evidence_density", "query_desc_len", "query_title_len"]
print("Project root:", L.ROOT)

## The modeling problem

The analysis notebook showed that feedback re-ranking is powerful but fragile. We now model the
*magnitude and sign* of the feedback bonus. A retrieval bonus is built from two ingredients:

1. **Evidence**: how often, and how positively, a candidate was judged useful within a scope.
2. **A lift formula**: how that evidence is converted into a bounded bonus added to the retrieval score.

The two modeling levers we study are **where the formula is centred** (what counts as "neutral"
feedback) and **how large the bonus is relative to the retriever's score scale**. We also test which
scope's evidence to use (evidence discipline).

### Which lift formulas exist, and what are their failure modes?

**What we do.** Implementations: Laplace, empirical-Bayes Laplace, tanh, and Bayesian lower-confidence-bound. We inspect how each maps evidence to a bonus.

**Artifact.** `src/feedback/lift.py; results/feedback_calibration/saturation.csv`

**Caveat.** tanh and LCB are implemented but not yet evaluated on data (flagged).

*What this cell does.* Plot each formula's output as a function of net evidence, then read the empirical saturation table for Laplace vs empirical-Bayes.

In [ ]:
import numpy as np
pos = np.linspace(0, 10, 200)
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(pos - 5, (pos + 1) / (pos + 5 + 2) - 0.5, label="laplace (centred 0.5)")
ax.plot(pos - 5, np.tanh(pos - 5) * 0.2, label="tanh")
ax.axhline(0, color="black", lw=1); ax.set(xlabel="Pos - Neg (net evidence)", ylabel="Raw bonus", title="Lift formula shapes")
ax.legend(); plt.tight_layout(); L.savefig("04_lift_shapes", run_ids=[]); plt.show()

sat = L.load_calib_saturation()
display(sat.sort_values(["routing", "lift"])[["lift", "routing", "pct_at_minus_cap", "pct_at_plus_cap",
        "pct_zero", "n_distinct_lift_values", "lift_std_over_faiss_std"]].round(3))

**Reading.** Laplace centred at 0.5 is *linear* in evidence and symmetric, so with a corpus whose
judge scores are mostly below 0.5 it pushes most candidates negative. The saturation table confirms
the practical consequence: for the legacy Laplace lift, 40–50% of pool bonuses sit at the negative
cap, meaning the formula cannot distinguish "bad" from "very bad". Empirical-Bayes centering
(developed next) reduces cap saturation to a few percent and produces far more distinct values. tanh
and the LCB are implemented for completeness but are flagged as not yet evaluated, so we do not
claim results for them.

### Modeling choice 1 — centre the formula on the scope's own base rate

**What we do.** Replace the fixed 0.5 neutral point with each scope's empirical mean judge score (empirical-Bayes), so that average evidence yields zero bonus.

**Artifact.** `results/feedback_calibration/saturation.csv; results/retriever_ladder/grid.csv`

**Caveat.** Prior strength κ and cap remain fixed.

*What this cell does.* Show the heterogeneous scope base rates, then compare legacy Laplace against empirical-Bayes on the conditioned ladder per routing.

In [ ]:
priors = L.load_calib_scope_priors("conditioned").copy()
priors["type"] = priors["scope"].str.split(":").str[0]
fig, ax = plt.subplots(figsize=(9, 4.5)); sns.histplot(data=priors, x="mean", hue="type", element="step", common_norm=False, ax=ax)
ax.set(xlabel="Observed scope base rate", title="Scope base rates are heterogeneous")
plt.tight_layout(); L.savefig("04_scope_base_rates", run_ids=[]); plt.show()

grid = L.load_ladder_grid(); metric = "minilm_d_proxy_top1"
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=grid[grid["lift"].isin(["laplace|abs", "laplace_eb_k2|abs", "laplace_eb_k10|abs"])],
            x="routing", y=metric, hue="lift", ax=ax)
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Best proxy gain", title="Centering: Laplace vs empirical-Bayes (conditioned, absolute)")
plt.tight_layout(); L.savefig("04_centering_ablation", run_ids=[]); plt.show()

**Reading.** Scope base rates range from near 0 to about 0.55, so a single 0.5 centre is wrong for
almost every scope. Centering on the scope's own mean turns "average evidence" into "no bonus" and
lets strong evidence stand out. On the conditioned ladder, empirical-Bayes improves the broad
routings (which legacy Laplace drove negative) while leaving the fine scopes competitive. The
practical effect is largest where the fixed centre was most wrong.

### Modeling choice 2 — express the bonus in the retriever's own score units

**What we do.** Absolute bonuses add a fixed value to cosine scores; pool-relative scaling multiplies the bounded bonus by the candidate pool's score standard deviation, so the same cap means the same relative shift across retrievers.

**Artifact.** `results/retriever_ladder_blind/grid.csv`

**Caveat.** Offline proxy evidence; generated confirmation follows.

*What this cell does.* Group the blind ladder by scaling mode and show the best gain per scaling, then the per-retriever best cells.

In [ ]:
blind = L.load_ladder_grid("blind"); metric = "minilm_d_proxy_top1"
blind = blind.assign(scale=blind["lift"].str.split("|").str[1])
best_by_scale = blind.groupby("scale")[metric].max().sort_values(ascending=False)
display(best_by_scale.round(4))
fig, ax = plt.subplots(figsize=(9, 4.5)); best_by_scale.plot.bar(ax=ax, color="#16856b")
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Best blind proxy gain", xlabel="Scaling mode",
                                           title="Pool-relative scaling rescues ticket-only feedback")
plt.tight_layout(); L.savefig("04_scaling_rescue", run_ids=[]); plt.show()
best = blind.loc[blind.groupby("retriever")[metric].idxmax()][["retriever", "routing", "lift", metric, "primary_ci_lo", "primary_ci_hi"]]
display(best.round(4))

**Reading.** Under ticket-only feedback, absolute bonuses are harmful (the bonus is the wrong size
for the score distribution), whereas pool-relative scaling — which shrinks the bonus when the pool's
scores are tightly packed — turns the same evidence into a positive gain. This is the single most
important modeling result: the failure of realistic feedback is a *scaling* failure, not a signal
failure, and it is fixed by expressing the bonus relative to the retriever's own scale.

### Modeling choice 3 — discipline the evidence (hierarchical backoff)

**What we do.** Instead of a fixed scope, use the finest scope that has at least a minimum amount of evidence, falling back to broader scopes otherwise.

**Artifact.** `results/retriever_ladder{,_blind}/grid.csv; results/blend*/learned_weights.json`

**Caveat.** Minimum evidence is a hyperparameter set on dev.

*What this cell does.* Compare the best gain of a fixed intersection scope against backoff, for both protocols, then show the learned blend weights (the null result).

In [ ]:
for tag, label in [("", "conditioned"), ("blind", "blind")]:
    g = L.load_ladder_grid(tag)
    focus = g[g["routing"].isin(["M2_team", "M4_intersection", "M5_backoff"])].copy()
    best = focus.loc[focus.groupby(["retriever", "routing"])[metric].idxmax()]
    print(label)
    display(best.pivot_table(index="retriever", columns="routing", values=metric).round(4))
w = L.load_blend_weights("blend_eb")
display(pd.Series(w["weights"]).to_frame("learned_weight"))

**Reading.** Backoff matches or beats a fixed intersection scope because it keeps the benefit where
fine-scoped evidence exists and degrades gracefully where it does not — the common case in an
imbalanced corpus. The learned multi-scope blend collapses onto a single scope (near 1.0 on the
intersection weight, about 0 on the rest), i.e. the model discovers that combining scopes is
unnecessary; we keep it as a documented null result rather than a method.

### The aggression tradeoff — why one prior cannot serve both protocols

**What we do.** Plot the best achievable gain as a function of how aggressively the prior acts, for conditioned and ticket-only feedback.

**Artifact.** `results/retriever_ladder{,_blind}/grid.csv`

**Caveat.** This is the paper's central modeling statement.

*What this cell does.* For each scaling magnitude, show the best conditioned and blind gain, illustrating the tradeoff.

In [ ]:
rows = []
for tag, label in [("", "conditioned"), ("blind", "blind")]:
    g = L.load_ladder_grid(tag).copy()
    g["scale"] = g["lift"].str.split("|").str[1]
    for scale, part in g.groupby("scale"):
        rows.append({"protocol": label, "scale": scale, "best_gain": part[metric].max()})
tradeoff = pd.DataFrame(rows)
pivot = tradeoff.pivot(index="scale", columns="protocol", values="best_gain").sort_index()
display(pivot.round(4))
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot.bar(ax=ax, color={"conditioned": "#16856b", "blind": "#c44e52"})
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Best proxy gain", xlabel="Bonus scale (absolute or pool-std λ)",
                                           title="Aggression tradeoff across protocols")
plt.tight_layout(); L.savefig("04_aggression_tradeoff", run_ids=[]); plt.show()

**Reading.** Large absolute bonuses are best for resolution-informed feedback but catastrophic for
ticket-only feedback; pool-relative bonuses of moderate size are safe under both, at some cost to
the conditioned optimum. There is no single aggressiveness that is optimal for both protocols. This
motivates two legitimate responses: pick a conservative prior when feedback quality is unknown, or
learn a per-ticket control policy that sets the aggressiveness (notebook 05).

### Do the generated finalists confirm the prior modeling?

**What we do.** Compare generated dev results for legacy Laplace, empirical-Bayes, and the calibrated backoff on both protocols, with independent metrics.

**Artifact.** `results/rescored/method_comparison_v2.csv`

**Caveat.** Single dev seed; eval and robustness in notebook 06.

*What this cell does.* Print the generated finalist table with MiniLM, BGE, and BERTScore deltas and their p-values.

In [ ]:
resc = L.load_rescore_comparison()
cols = ["run", "delta_cosine_mean", "delta_cosine_wilcoxon_p", "delta_cosine_bge_mean",
        "delta_cosine_bge_wilcoxon_p", "delta_bertscore_f1_mean", "delta_bertscore_f1_wilcoxon_p"]
display(resc[[c for c in cols if c in resc]].round(4))

**Reading.** The generated runs reproduce the ladder's direction: legacy Laplace is strongly
negative under ticket-only feedback, the calibrated prior removes that harm and is mildly positive,
and resolution-informed runs remain positive. Independent metrics (BGE cosine, BERTScore) agree in
sign, so the effect is not an artefact of the MiniLM family used for retrieval. Magnitudes are
small and not individually significant on dev, which we state honestly.

## Recommended prior (method card, part 1)

Use the empirical-Bayes centered Laplace lift with a fixed cap, hierarchical backoff at minimum
evidence 2, and **pool-relative scaling** when the feedback's reliability is unknown. This is the
configuration carried into the control-policy notebook and the final results.